In [1]:
# pip install "lerobot[smolvla]"
# !source venv/bin/activate
# !pip install "lerobot[smolvla]"
# !pip install "lerobot[dataset]"
# !pip install --upgrade mistral-common


In [2]:
import torch

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import make_pre_post_processors

# Swap this import per-policy
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

W1113 00:18:30.591000 61590 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
objc[61590]: Class AVFFrameReceiver is implemented in both /Users/sanner/Coding/PEFT-Eval-on-VLAs/.pixi/envs/default/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x13e5dc3a8) and /Users/sanner/Coding/PEFT-Eval-on-VLAs/.pixi/envs/default/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x15b9243a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[61590]: Class AVFAudioReceiver is implemented in both /Users/sanner/Coding/PEFT-Eval-on-VLAs/.pixi/envs/default/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x13e5dc3f8) and /Users/sanner/Coding/PEFT-Eval-on-VLAs/.pixi/envs/default/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x15b9243f8). This may cause spurious casting f

In [3]:
import av
import torch

print("PyAV:", av.__version__)
print("PyTorch:", torch.__version__)

print("MPS available:", torch.backends.mps.is_available())



PyAV: 15.1.0
PyTorch: 2.10.0
MPS available: True


In [5]:
# load a policy
model_id = "lerobot/smolvla_base"  # <- swap checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy = SmolVLAPolicy.from_pretrained(model_id).to(device).eval()

print("\nInput shape:")
for key, feat in policy.config.input_features.items():
    print(f" - {key}: shape {feat.shape}")


preprocess, postprocess = make_pre_post_processors(
    policy.config,
    model_id,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)
# load a lerobotdataset
dataset = LeRobotDataset("lerobot/libero")

# pick an episode
episode_index = 0

# each episode corresponds to a contiguous range of frame indices
from_idx = dataset.meta.episodes["dataset_from_index"][episode_index]
to_idx   = dataset.meta.episodes["dataset_to_index"][episode_index]

# get a single frame from that episode (e.g. the first frame)
frame_index = from_idx
frame = dict(dataset[frame_index])

# 1. Remap LIBERO's camera keys to match smolvla_base's expected input_features
frame["observation.images.camera1"] = frame.pop("observation.images.image")
frame["observation.images.camera2"] = frame.pop("observation.images.image2")

# 2. Preprocess and pass `batch` (not `frame`) to select_action
batch = preprocess(frame)
with torch.inference_mode():
    pred_action = policy.select_action(batch)
    pred_action = postprocess(pred_action)

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Reducing the number of VLM layers to 16 ...

Input shape:
 - observation.state: shape (6,)
 - observation.images.camera1: shape (3, 256, 256)
 - observation.images.camera2: shape (3, 256, 256)
 - observation.images.camera3: shape (3, 256, 256)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]